You are given a dataset containing employee information, 
including department, salary, and years of experience. 
Write a PySpark program to get hight sal by department.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg

# Initialize Spark session
spark = SparkSession.builder.appName("DepartmentSalaryAnalysis").getOrCreate()

# Input data
data = [
    ("Alice", "HR", 50000, 5, "2024"),
    ("Bob", "Finance", 60000, 6, "2024"),
    ("Charlie", "HR", 55000, 4, "2024"),
    ("David", "Engineering", 90000, 8, "2024"),
    ("Eve", "Finance", 65000, 7, "2023"),
    ("Frank", "Engineering", 95000, 10, "2023"),
    ("Grace", "HR", 52000, 3, "2023")
]

columns = ["name", "department", "salary", "experience", "year"]

# Create DataFrame
df_Emp_Details = spark.createDataFrame(data, columns)
display(df_Emp_Details)
# 1. Calculate average salary per department
avg_salary_df = df.groupBy("department").agg(avg("salary").alias("avg_salary"))

print("Average Salary per Department:")
avg_salary_df.show()

# 2. Find department with highest average salary
highest_avg_salary = avg_salary_df.orderBy("avg_salary", ascending=False).limit(1)

print("Department with Highest Average Salary:")
highest_avg_salary.show()

In [0]:
data = [
        ("1", "sai", "10000", "6"),
        ("2", "Kiran", "9000", "7"),
        ("3", "Mantri", "7000", "null"),
        ("4", "Ravi", "8000", "Null"),
        ("5", "Rajesh", "6000", "null")        
        ]
Names = ["id", "name", "salary", "ManagerID"]

df = spark.createDataFrame(data, Names)
# df = df.agg(max("salary").alias("avg_salary"))
# display(df)

# from pyspark.sql.functions import col, row_number
# from pyspark.sql.window import Window

# window_spec = Window.partitionBy("ManagerID") \
#                     .orderBy(col("salary").cast("int").desc())

# result = (
#     df.withColumn("rn", row_number().over(window_spec))
#       .filter(col("rn") == 1)
#       .drop("rn")
# )

# result.show()
df = df.createOrReplaceTempView("employee")
df = spark.sql("""SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER(
               PARTITION BY ManagerID
               ORDER BY CAST(salary AS INT) DESC
           ) rn
    FROM employee
)
WHERE rn = 1 limit(1) """)
display(df)

#df.printSchema()
# display(df)

df.createOrReplaceTempView("Employee")
# DBTITLE 1,SQL
# df = spark.sql("""SELECT e.id,       e.name,
#        e.salary,
#        e.ManagerID
# FROM Employee e
# CROSS JOIN Employee m
#      ON e.id = m.id
# WHERE e.salary > m.salary""")
# display(df)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg
from pyspark.sql.functions import max

# Initialize Spark session
spark = SparkSession.builder.appName("DepartmentSalaryAnalysis").getOrCreate()

# Input data
data = [
    ("Alice", "HR", 50000, 5, "2024"),
    ("Bob", "Finance", 60000, 6, "2024"),
    ("Charlie", "HR", 55000, 4, "2024"),
    ("David", "Engineering", 90000, 8, "2024"),
    ("Eve", "Finance", 65000, 7, "2023"),
    ("Frank", "Engineering", 95000, 10, "2023"),
    ("Grace", "HR", 52000, 3, "2023")
]

columns = ["name", "department", "salary", "experience", "year"]

df = spark.createDataFrame(data, columns)
#display(df)

# df_Dist_Dept = df.groupby("department").agg(avg("salary").alias("avg_sal"))
# df_Dist_Dept = df.groupBy("department").agg(max("salary").alias("avg_sal"))
df_Dist_Dept = df.groupBy("department").agg(max("salary").alias("avg_sal"))\
                .orderBy("avg_sal", ascending=False).limit(1)
display(df_Dist_Dept)

# df_Highest_Sal = df_Dist_Dept.orderBy("avg_sal", ascending=False).limit(1)
# display(df_Highest_Sal)

# df_Highest_Sal.createOrReplaceTempView("Highest_Sal")
# df_Highest_Sal = spark.sql("""SELECT max(avg_sal)
# FROM Highest_Sal""")
# display(df_Highest_Sal)

# df = df.createOrReplaceTempView("GetMaxSal")
# df_max = spark.sql("""SELECT name,
#        department,
#        salary
# FROM (
#     SELECT *,
#            ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS rn
#     FROM GetMaxSal
# ) t
# WHERE rn = 1 LIMIT 1;""")
# display(df_max)




In [0]:
# Hight salary per department
# display(df_Emp_Details)
# df = df_Emp_Details.groupBy("department").agg(max("salary").alias("avg_sal"))
# df = df.orderBy("avg_sal", ascending=False).limit(1)
# display(df)

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())
result = df_Emp_Details.withColumn("row", row_number().over(window_spec)) \
          .filter(col("row") == 1).drop("row").limit(1)
display(result)

df = df_Emp_Details.createOrReplaceTempView("EmployeeDel")
df = spark.sql("""
               SELECT * FROM 
               (
                    SELECT *,
                    ROW_NUMBER() OVER
                    ( 
                        PARTITION BY department ORDER BY salary DESC
                    ) AS rn
                    FROM EmployeeDel
                ) t
               WHERE rn = 1 limit 1               
               """)

display(df)




# # Lowest salary per department
# from pyspark.sql.functions import max, min
# df = df_Emp_Details.groupBy("department").agg(min("salary").alias("Min_sal"))
# df = df.orderBy("Min_sal", decending=False).limit(1)
# display(df)

# # Avg for all employess
# df = df_Emp_Details.agg(avg("salary").alias("avg_sal"))
# display(df)


In [0]:
df =  (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", "abfss://source@mantristorageaccount.dfs.core.windows.net/metadata/schema/claims")
    .load("abfss://source@mantristorageaccount.dfs.core.windows.net/claims")
)
display(df, checkpointLocation="abfss://source@mantristorageaccount.dfs.core.windows.net/metadata/checkpoints/claims")

In [0]:
df_Approved_Claims = df.filter(df.Status == "Approved")
#display(df_Approved_Claims)
df_count = df_Approved_Claims.groupBy("ProviderID","MemberID")\
            .agg(avg("ClaimAmount").alias("AvgClaimAmount")) \
                .withColumn("AvgClaimAmount",col("AvgClaimAmount").cast("double")) 

df_BetweenDates = df_Approved_Claims.filter(df_Approved_Claims.ClaimDate >= "2026-07-15")\
                                    .filter(df_Approved_Claims.ClaimDate <= "2026-07-16")
#display(df_BetweenDates)

df_Approved_Claims.writeStream.format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "abfss://bronze@mantristorageaccount.dfs.core.windows.net/claims") \
    .toTable("healthcare.silver.claims_Approved_bronze")
display(df_Approved_Claims, checkpointLocation = "abfss://bronze@mantristorageaccount.dfs.core.windows.net/claims")

# display(df_count)

In [0]:
%sql
describe history healthcare.bronze.claims_new

In [0]:
df_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", "abfss://bronze@mantristorageaccount.dfs.core.windows.net/metadata/schemas/claims")
    .load("abfss://source@mantristorageaccount.dfs.core.windows.net/claims")
)

df_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://bronze@mantristorageaccount.dfs.core.windows.net/metadata/checkpoints/claims") \
    .toTable("healthcare.bronze.claims")